In [1]:
# @title
!pip install s3fs
!pip install pyarrow
!pip install "dask[complete]"
!pip install memory-profiler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.7 MB/s eta 0:00:00


In [2]:
from datetime import datetime
import dask.dataframe as dd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import s3fs # Ensure s3fs is imported for S3 access
import dask.dataframe as dd # Import dask.dataframe
%load_ext memory_profiler

**Loading the Data into a Dask Dataframe**

In [3]:
import dask.dataframe as dd

base_s3_uri = "s3://data604-g4-final/Combined_Data/run-1774669705550-part-block-0-r-"
runs = [f"{i:05d}" for i in range(123)]

s3_urls = [f"{base_s3_uri}{r}-uncompressed.parquet" for r in runs]

df = dd.read_parquet(s3_urls, storage_options={"anon": True})

In [4]:
df.head()

,day_of_week,fl_date,op_unique_carrier,op_carrier_airline_id,op_carrier,tail_num,op_carrier_fl_num,origin_airport_id,origin_airport_seq_id,origin_city_market_id,...,origin_weather_code,origin_wind_speed_10m,origin_wind_gusts_10m,dest_temperature_2m,dest_precipitation,dest_rain,dest_snowfall,dest_weather_code,dest_wind_speed_10m,dest_wind_gusts_10m
0,2,6/18/2024 12:00:00 AM,9E,20363,9E,N921XJ,5232,10397,1039707,30397,...,3.0,25.849905,47.519997,32.95,0.0,0.0,0.0,3.0,21.897945,47.519997
1,2,6/18/2024 12:00:00 AM,9E,20363,9E,N935XJ,5072,12478,1247805,31703,...,1.0,20.833395,39.6,31.35,0.0,0.0,0.0,0.0,12.245294,30.960001
2,2,6/18/2024 12:00:00 AM,AA,19805,AA,N402AN,1138,11057,1105703,31057,...,3.0,6.5693827,12.24,17.45,0.0,0.0,0.0,0.0,5.495161,8.64
3,2,6/18/2024 12:00:00 AM,AA,19805,AA,N449AN,1147,14107,1410702,30466,...,0.0,10.446206,24.84,28.1,0.0,0.0,0.0,0.0,17.462784,35.28
4,2,6/18/2024 12:00:00 AM,B6,20409,B6,N3044J,285,12478,1247805,31703,...,1.0,20.833395,39.6,31.35,0.0,0.0,0.0,0.0,12.245294,30.960001


**Preparing the Data for Modelling**

In [5]:
# converting date column to datetime
df["fl_date"] = dd.to_datetime(df["fl_date"], format='%m/%d/%Y %I:%M:%S %p')

In [6]:
# creating a new month column
df["month"] = df["fl_date"].dt.month

In [7]:
# checks counts of cancelled versus not cancelled fligts
df.cancelled.value_counts().compute()

,count
cancelled,
0.00,837176
1.00,13046


In [8]:
# dropping flights that were cancelled
# these flights do not have a value for arrival delay
df = df[df["cancelled"] != "1.00"]

In [9]:
# drops rows with arrival delay missing
df = df[df["arr_delay"] != ""]

In [10]:
# checking for empty strings in the data
blank_counts = (df == '').sum()
blank_counts.compute()

,0
day_of_week,0.0
fl_date,0.0
op_unique_carrier,0.0
op_carrier_airline_id,0.0
op_carrier,0.0
tail_num,0.0
op_carrier_fl_num,0.0
origin_airport_id,0.0
origin_airport_seq_id,0.0
origin_city_market_id,0.0


In [11]:
# checking for missing values
df.isna().sum().compute()

,0
day_of_week,0
fl_date,0
op_unique_carrier,0
op_carrier_airline_id,0
op_carrier,0
tail_num,0
op_carrier_fl_num,0
origin_airport_id,0
origin_airport_seq_id,0
origin_city_market_id,0


In [14]:
# converts columns with numeric variables to numeric data type
columns_to_convert = ['crs_dep_time','dep_time', 'dep_delay', 'crs_arr_time', 'arr_time', 'arr_delay', 'air_time', 'distance', 'origin_temperature_2m', 'origin_precipitation', 'origin_rain', 'origin_snowfall', 'origin_wind_speed_10m', 'origin_wind_gusts_10m', 'dest_temperature_2m', 'dest_precipitation', 'dest_rain', 'dest_snowfall', 'dest_wind_speed_10m', 'dest_wind_gusts_10m']
for col in columns_to_convert:
    df[col] = dd.to_numeric(df[col], errors='coerce')

In [15]:
# converts columns with categorical variables to category data type
df[["day_of_week", 'op_carrier', 'origin', 'dest', 'origin_weather_code', 'dest_weather_code', 'month']] = df[["day_of_week", 'op_carrier', 'origin', 'dest', 'origin_weather_code', 'dest_weather_code', 'month']].astype("category")
df.dtypes

,0
day_of_week,category
fl_date,datetime64[ns]
op_unique_carrier,string[pyarrow]
op_carrier_airline_id,string[pyarrow]
op_carrier,category
tail_num,string[pyarrow]
op_carrier_fl_num,string[pyarrow]
origin_airport_id,string[pyarrow]
origin_airport_seq_id,string[pyarrow]
origin_city_market_id,string[pyarrow]


In [16]:
# drops columns that will not be used in the model
df = df.drop(columns = ["fl_date", "op_unique_carrier", "op_carrier_airline_id", "tail_num", "op_carrier_fl_num", "origin_airport_id", "origin_airport_seq_id", "origin_city_market_id", "origin_city_name", "origin_state_abr", "dest_airport_id", "dest_airport_seq_id", "dest_city_market_id", "dest_city_name", "dest_state_abr", "dep_time", "dep_delay", "arr_time", "cancelled", "cancellation_code", "diverted", "air_time", "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay", "fl_date_key", "dep_hour_key", "arr_hour_key"])

In [17]:
# checks the type of data for the remaining columns
df.dtypes

,0
day_of_week,category
op_carrier,category
origin,category
dest,category
crs_dep_time,Int64
crs_arr_time,Int64
arr_delay,Int64
distance,Int64
origin_temperature_2m,Int64
origin_precipitation,Int64


In [18]:
# drops any remaing missing values
df = df.dropna()
# ensures there are no longer any missing values
df.isna().sum().compute()

,0
day_of_week,0
op_carrier,0
origin,0
dest,0
crs_dep_time,0
crs_arr_time,0
arr_delay,0
distance,0
origin_temperature_2m,0
origin_precipitation,0


**Model Training**

In [ ]:
# splits the dask dataframe into X (predictor variables) and y (the target variable)
y = df["arr_delay"]
X = df.drop(columns = "arr_delay")

In [ ]:
# splits the data into a train and test set
X_train, X_test, y_train, y_test = train_test_split(X.compute(), y.compute(), test_size = 0.2, random_state = 42)

In [ ]:
%%memit
# fits the train set to an XGBoost model
xg_model = XGBRegressor(enable_categorical = True)
xg_model.fit(X_train, y_train)

peak memory: 13561.09 MiB, increment: 2249.35 MiB


In [ ]:
xg_model.save_model("model.json")

Peak RAM usage with dask was calculated to be about 13.2 GB.

**Model Evaluation**

In [ ]:
# uses the trained model to make predictions on the test set
y_pred = xg_model.predict(X_test)
# evaluates the performance of the model on making predictions on the test set using mean absolute error
mae = mean_absolute_error(y_test, y_pred)
print(mae)

25.56243324279785
